# HDB CatBoost Model Training Notebook

This notebook trains the same CatBoost model used by the deployment project. It is useful for coursework/demo explanation, while `train_model.py` remains the production training script.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error

from data_loader import fetch_hdb_resale_data, clean_hdb_data
from train_model import FEATURES, TARGET, CAT_FEATURES, MODEL_DIR, MODEL_PATH, METRICS_PATH

In [ ]:
# For quick notebook testing, use 10000 records.
# For final training, change max_records=None.
raw = fetch_hdb_resale_data(max_records=10000)
data = clean_hdb_data(raw)
print(data.shape)
data[FEATURES + [TARGET]].head()

In [ ]:
X = data[FEATURES]
y = np.log1p(data[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_pool = Pool(X_train, y_train, cat_features=CAT_FEATURES)
test_pool = Pool(X_test, y_test, cat_features=CAT_FEATURES)

In [ ]:
model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    early_stopping_rounds=80,
    verbose=100,
)

model.fit(train_pool, eval_set=test_pool)

In [ ]:
pred = np.expm1(model.predict(X_test))
actual = np.expm1(y_test)

metrics = {
    "model_type": "CatBoostRegressor",
    "records_used": int(len(data)),
    "mae_sgd": float(mean_absolute_error(actual, pred)),
    "rmse_sgd": float(root_mean_squared_error(actual, pred)),
    "mape": float(mean_absolute_percentage_error(actual, pred)),
    "r2": float(r2_score(actual, pred)),
    "features": FEATURES,
    "cat_features": CAT_FEATURES,
}
metrics

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(actual, pred, alpha=0.25)
plt.title("Actual vs Predicted HDB Resale Price")
plt.xlabel("Actual Price SGD")
plt.ylabel("Predicted Price SGD")
plt.show()

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
model.save_model(str(MODEL_PATH))
joblib.dump(metrics, METRICS_PATH)
print(f"Saved model to {MODEL_PATH}")
print(f"Saved metrics to {METRICS_PATH}")